# RGB → Multispectral Prediction — WeedyRice Dataset
**Run cells top to bottom. Everything is automatic.**

Before starting:
- Runtime → Change runtime type → **T4 GPU**
- Make sure your zip is on Google Drive

In [ ]:
# ============================================================
# CELL 1 — CONFIGURE PATHS (only cell you need to edit)
# ============================================================

# ---- Set these two lines to match your Drive ----
ZIP_PATH      = '/content/drive/MyDrive/FYI/WeedyRice-RGBMS-DB (1).zip'
PROJECT_DIR   = '/content/drive/MyDrive/rgb2ms'          # where code lives

# ---- These are fixed — do not change ----
EXTRACT_DIR   = '/content/dataset'                        # local fast SSD
CHECKPOINT_DIR= '/content/drive/MyDrive/rgb2ms_checkpoints'
EVAL_DIR      = '/content/drive/MyDrive/rgb2ms_eval'
MODELS_DIR    = '/content/drive/MyDrive/rgb2ms_models'

print('Paths configured.')
print(f'  ZIP      : {ZIP_PATH}')
print(f'  Extract  : {EXTRACT_DIR}')
print(f'  Code     : {PROJECT_DIR}')
print(f'  Checkpts : {CHECKPOINT_DIR}')

In [ ]:
# ============================================================
# CELL 2 — MOUNT DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ============================================================
# CELL 3 — INSTALL DEPENDENCIES
# ============================================================
!pip install -q tifffile onnx onnxruntime
print('Dependencies installed.')

In [ ]:
# ============================================================
# CELL 4 — EXTRACT DATASET (once; skips if already done)
# ============================================================
import os, zipfile, pathlib

# ---- Auto-detect the dataset root inside the zip ----
def find_dataset_root(base_dir: str) -> str:
    """
    Walk the extracted directory to find the folder that contains
    RGB/, Multispectral/, Metadata/ — that is the dataset root.
    """
    for root, dirs, files in os.walk(base_dir):
        subdirs = set(dirs)
        if {'RGB', 'Multispectral', 'Metadata'}.issubset(subdirs):
            return root
    raise RuntimeError(
        f'Could not find RGB/ + Multispectral/ + Metadata/ inside {base_dir}\n'
        f'Top-level contents: {os.listdir(base_dir)}'
    )

# ---- Check if already extracted ----
marker = os.path.join(EXTRACT_DIR, '.extracted')
if os.path.exists(marker):
    print('Dataset already extracted — skipping unzip.')
else:
    print(f'Extracting {ZIP_PATH} ...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            f'Zip not found: {ZIP_PATH}\n'
            f'Check CELL 1 — make sure ZIP_PATH matches your Drive exactly.'
        )

    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

    # Write marker so we skip next time
    pathlib.Path(marker).touch()
    print('Extraction complete.')

# ---- Find and store the exact dataset root ----
DATA_ROOT = find_dataset_root(EXTRACT_DIR)
print(f'\nDataset root : {DATA_ROOT}')
print(f'Contents     : {os.listdir(DATA_ROOT)}')

# Verify expected subfolders
for sub in ['RGB', 'Multispectral', 'Metadata', 'Masks']:
    p = os.path.join(DATA_ROOT, sub)
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    status = '✓' if n > 0 else '✗ MISSING'
    print(f'  {status}  {sub}/  ({n} files)')

# Save DATA_ROOT so other cells can read it without re-running
with open('/content/data_root.txt', 'w') as f:
    f.write(DATA_ROOT)
print(f'\nDATA_ROOT saved to /content/data_root.txt')

In [ ]:
# ============================================================
# CELL 5 — SET UP CODE DIRECTORY
# ============================================================
import os, sys

os.makedirs(PROJECT_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(PROJECT_DIR)}')

In [ ]:
# ============================================================
# CELL 6 — VERIFY GPU
# ============================================================
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ============================================================
# CELL 7 — COMPUTE DATASET NORMALIZATION STATS
# Run once. Paste output into dataset.py and rpi_inference.py
# ============================================================

# Read DATA_ROOT (works even if Cell 4 was not re-run this session)
with open('/content/data_root.txt') as f:
    DATA_ROOT = f.read().strip()
print(f'Dataset root: {DATA_ROOT}')

!python compute_stats.py --data "{DATA_ROOT}"

In [ ]:
# ============================================================
# CELL 8 — UPDATE NORMALIZATION STATS IN CODE
# Paste the values printed by Cell 7 below, then run this cell.
# ============================================================

# ---- PASTE YOUR COMPUTED VALUES HERE ----
RGB_MEAN = [0.485, 0.456, 0.406]   # <-- replace with Cell 7 output
RGB_STD  = [0.229, 0.224, 0.225]   # <-- replace
MS_MEAN  = [0.150, 0.120, 0.180, 0.350]  # [Green, Red, RedEdge, NIR] <-- replace
MS_STD   = [0.080, 0.070, 0.090, 0.150]  # <-- replace

import re

def patch_stats(filepath, rgb_mean, rgb_std, ms_mean, ms_std):
    with open(filepath) as f:
        src = f.read()
    src = re.sub(r'RGB_MEAN\s*=\s*\[.*?\]', f'RGB_MEAN = {rgb_mean}', src)
    src = re.sub(r'RGB_STD\s*=\s*\[.*?\]',  f'RGB_STD  = {rgb_std}',  src)
    src = re.sub(r'MS_MEAN\s*=\s*\[.*?\]',  f'MS_MEAN  = {ms_mean}',  src)
    src = re.sub(r'MS_STD\s*=\s*\[.*?\]',   f'MS_STD   = {ms_std}',   src)
    with open(filepath, 'w') as f:
        f.write(src)
    print(f'  Patched: {filepath}')

patch_stats(f'{PROJECT_DIR}/dataset.py',       RGB_MEAN, RGB_STD, MS_MEAN, MS_STD)
patch_stats(f'{PROJECT_DIR}/rpi_inference.py', RGB_MEAN, RGB_STD, MS_MEAN, MS_STD)
print('Stats updated in dataset.py and rpi_inference.py')

In [ ]:
# ============================================================
# CELL 9 — TRAIN
# Saves checkpoint to Drive every 5 epochs.
# If Colab disconnects, re-run from Cell 2 and add --resume.
# ============================================================
import os

with open('/content/data_root.txt') as f:
    DATA_ROOT = f.read().strip()

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Check if a previous checkpoint exists to resume from
latest = f'{CHECKPOINT_DIR}/latest.pth'
resume_flag = f'--resume "{latest}"' if os.path.exists(latest) else ''
print(f'Resume flag: {resume_flag or "(fresh start)"}')

!python train.py \
    --data "{DATA_ROOT}" \
    --out  "{CHECKPOINT_DIR}" \
    --epochs   100 \
    --batch      8 \
    --img_size 512 \
    --workers    2 \
    {resume_flag}

In [ ]:
# ============================================================
# CELL 10 — EVALUATE ON TEST SET
# ============================================================
import os

with open('/content/data_root.txt') as f:
    DATA_ROOT = f.read().strip()

os.makedirs(EVAL_DIR, exist_ok=True)

!python evaluate.py \
    --data "{DATA_ROOT}" \
    --ckpt "{CHECKPOINT_DIR}/best_model.pth" \
    --out  "{EVAL_DIR}" \
    --save_vis

In [ ]:
# ============================================================
# CELL 11 — SHOW RESULTS
# ============================================================
import json
from IPython.display import Image as IPImage, display
import os

# Print metrics
metrics_path = f'{EVAL_DIR}/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print('Per-band R² results:')
    print('-' * 40)
    for band, vals in m.items():
        if band == 'mean_R2':
            print(f'\n  Mean R² = {vals:.4f}')
        else:
            r2   = vals['R2']
            flag = '✓ PASS' if r2 >= 0.75 else '✗ FAIL'
            print(f'  {band:10s}: R²={r2:.4f}  {flag}  '
                  f'RMSE={vals["RMSE"]:.5f}')

# Show scatter plots
scatter = f'{EVAL_DIR}/scatter_plots.png'
if os.path.exists(scatter):
    display(IPImage(scatter))

# Show first visual comparison
vis_dir = f'{EVAL_DIR}/visuals'
if os.path.exists(vis_dir):
    pngs = sorted(os.listdir(vis_dir))[:2]
    for p in pngs:
        display(IPImage(os.path.join(vis_dir, p)))

In [ ]:
# ============================================================
# CELL 12 — EXPORT TO ONNX + INT8 FOR RASPBERRY PI
# ============================================================
import os
os.makedirs(MODELS_DIR, exist_ok=True)

!python export_onnx.py \
    --ckpt     "{CHECKPOINT_DIR}/best_model.pth" \
    --out      "{MODELS_DIR}" \
    --img_size 512

print(f'\nDeploy this file on your Raspberry Pi:')
print(f'  {MODELS_DIR}/rgb2ms_int8.onnx')
print(f'\nAlso copy rpi_inference.py to the Pi.')

In [ ]:
# ============================================================
# CELL 13 — QUICK TEST INFERENCE (in Colab, using ONNX model)
# Pick one test image and predict its MS bands.
# ============================================================
import os, numpy as np
from PIL import Image
import onnxruntime as ort
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

with open('/content/data_root.txt') as f:
    DATA_ROOT = f.read().strip()

# Load normalization stats from dataset.py
import sys; sys.path.insert(0, PROJECT_DIR)
from dataset import RGB_MEAN, RGB_STD, MS_MEAN, MS_STD

INT8_MODEL = f'{MODELS_DIR}/rgb2ms_int8.onnx'
BAND_NAMES = ['Green', 'Red', 'RedEdge', 'NIR']

# Pick first image from test.txt
with open(f'{DATA_ROOT}/Metadata/test.txt') as f:
    test_stem = f.readline().strip().replace('.JPG','').replace('.jpg','')

rgb_path = os.path.join(DATA_ROOT, 'RGB', test_stem + '.JPG')
print(f'Test image: {os.path.basename(rgb_path)}')

# Preprocess
SIZE = 512
img  = np.array(Image.open(rgb_path).convert('RGB').resize((SIZE, SIZE))).astype(np.float32) / 255.0
img  = (img - np.array(RGB_MEAN)) / np.array(RGB_STD)
inp  = img.transpose(2,0,1)[np.newaxis].astype(np.float32)  # (1,3,H,W)

# Infer
sess = ort.InferenceSession(INT8_MODEL, providers=['CPUExecutionProvider'])
out  = sess.run(['ms'], {'rgb': inp})[0][0]   # (4, H, W)

# Denormalize
ms_mean = np.array(MS_MEAN)[:,None,None]
ms_std  = np.array(MS_STD)[:,None,None]
ms      = np.clip(out * ms_std + ms_mean, 0, 1)

# Plot
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
rgb_vis = np.array(Image.open(rgb_path).convert('RGB').resize((SIZE, SIZE)))
axes[0].imshow(rgb_vis);               axes[0].set_title('RGB Input')
cmaps = ['Greens','Reds','Oranges','RdPu']
for b in range(4):
    axes[b+1].imshow(ms[b], cmap=cmaps[b], vmin=0, vmax=1)
    axes[b+1].set_title(f'Pred {BAND_NAMES[b]}')
for ax in axes: ax.axis('off')
plt.tight_layout()
out_path = '/content/quick_test.png'
plt.savefig(out_path, dpi=120); plt.close()
display(IPImage(out_path))
print('NDVI:', ((ms[3]-ms[1])/(ms[3]+ms[1]+1e-8)).mean().round(4))

## Done!

Files saved to your Drive:
- `rgb2ms_checkpoints/best_model.pth` — PyTorch checkpoint
- `rgb2ms_models/rgb2ms_int8.onnx` — **copy this to Raspberry Pi**
- `rgb2ms_eval/metrics.json` — per-band R², RMSE, PSNR
- `rgb2ms_eval/scatter_plots.png` — predicted vs actual

### On Raspberry Pi:
```bash
pip install onnxruntime numpy Pillow tifffile
python rpi_inference.py --model rgb2ms_int8.onnx --image field.jpg
```